# 15. TSP
TSP (Traveling Salesman Problem) とは、都市の集合と都市間の距離が与えられたとき、すべての都市を一度ずつ訪れて出発点に戻る最短経路を求める問題です。TSPはNP困難な問題であり、効率的なアルゴリズムが存在しないため、近似アルゴリズムやヒューリスティックな手法がよく使用されます。ここでは、小さな問題の厳密解、大きな問題の簡易近似解、そしてモンテカルロ法を用いた近似解の例を示します。

TSP (Traveling Salesman Problem) is a problem where, given a set of cities and the distances between them, the task is to find the shortest possible route that visits each city exactly once and returns to the starting point. TSP is an NP-hard problem, meaning that there is no known efficient algorithm to solve it for large instances. Therefore, approximation algorithms and heuristic methods are often used. Here, we will show examples of an exact solution for small problems, a simple approximation solution for larger problems, and an approximation solution using the Monte Carlo method.

-----

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import itertools
import math

plt.rcParams['mathtext.fontset'] = 'cm'
    

## 小さな問題
都市数$n$が小さい、例えば$n=6$問題を定義します。都市を$[v_0, v_1, v_2, v_3, v_4, v_5]$とし、その間の距離を適当に設定します。都市間の距離は対称、つまり$d(v_i, v_j) = d(v_j, v_i)$とします。実際の問題では、$v_i$と$v_j$の間が直接接続されていない場合があるでしょう。その場合には、その距離を非常に大きい値としておくことにします。

経路の総数は$n!$ですが、出発点を固定することで、$(n-1)!$通りの経路を調べ、最も短い経路を見つけることにします。逆回りの距離も同じですが、まとめて調べることにします。

Let's define a small problem with $n=6$ cities. We will denote the cities as $[v_0, v_1, v_2, v_3, v_4, v_5]$ and set the distances between them appropriately. The distances between cities are symmetric, meaning that $d(v_i, v_j) = d(v_j, v_i)$. In real problems, there may be cases where there is no direct connection between $v_i$ and $v_j$. In such cases, we can set the distance to a very large value.


In [ ]:
def small_problem():
    """
    5頂点の小さな重み付きグラフを作成する。
    Create a small graph with 5 nodes and weighted edges.
    """
    g = nx.Graph()
    cityes = ['A', 'B', 'C', 'D', 'E']
    edges = [
        ('A', 'B', 10),
        ('A', 'C', 15),
        ('A', 'D', 20),
        ('A', 'E', 30),
        ('B', 'C', 30),
        ('B', 'D', 35),
        ('B', 'E', 40),
        ('C', 'D', 45),
        ('C', 'E', 50),
        ('D', 'E', 55),]
    # 重みとともに、辺をグラフに追加する。: Add edges to the graph with weights.
    g.add_weighted_edges_from(edges)
    # 頂点の座標を設定する。: Set the coordinates of the nodes.
    position = {
        'A': (0, 0),
        'B': (1, 0.3),
        'C': (2, 0),
        'D': (0.5, 1),
        'E': (1.5, 1)}
    nx.set_node_attributes(g, position, 'pos')
    return g

def find_tour(g): 
    """
    すべての可能な巡回路を探索し、最良のものを返す。
    Search all possible tours and return the best one.
    """
    min_cost = float('inf')# 最小コストを初期化する。: Initialize the minimum cost.
    best_tour = None
    # 頂点のすべての順列のイテレータを作成する。
    # Create an iterator for all permutations of the nodes in the graph.
    permutations = itertools.permutations(g.nodes())
    for tour in permutations:
        cost = 0
        for i in range(len(tour)):
            # i番目の頂点と次の頂点の間のコストを加算する。
            # Add the cost between the i-th node and the next node.
            u = tour[i]
            # 次の頂点は、最後の頂点の場合は最初の頂点に戻る。
            # The next node is the first node if the current node is the last one.
            v = tour[(i + 1) % len(tour)]
            cost += g[u][v]['weight']
        if cost < min_cost:
            # もし現在の巡回路のコストが最小コストより小さい場合、最小コストと最良の巡回路を更新する。
            # If the cost of the current tour is less than the minimum cost, update the minimum
            min_cost = cost
            best_tour = tour
    return best_tour, min_cost

g = small_problem()
pos = nx.get_node_attributes(g, 'pos')
weight_labels = nx.get_edge_attributes(g, 'weight')
nx.draw(g, pos=pos, with_labels=True)
nx.draw_networkx_edge_labels(g, pos=pos, edge_labels=weight_labels)
best_tour, min_cost = find_tour(g)
print("Best tour:", best_tour)
print("Minimum cost:", min_cost)
edges_in_best_tour = [(best_tour[i], best_tour[(i + 1) % len(best_tour)]) for i in range(len(best_tour))]
nx.draw_networkx_edges(g, pos=pos, edgelist=edges_in_best_tour, edge_color='r', width=2)
plt.show()

## 階乗の怖さ
階乗は非常に急速に増加する。そのため、都市数が増えると確認すべき経路の数が爆発的に増加し、すべてを調べることができなくなる。

Factorials grow very rapidly. Therefore, as the number of cities increases, the number of routes to check increases explosively, making it impossible to examine all of them.

In [ ]:
x = [n for n in range(2, 33)]
y = [math.factorial(n) for n in x]
plt.plot(x, y)
plt.yscale('log')
plt.xlabel('$n$')
plt.ylabel('$n!$')
plt.xlim(0, 35)
plt.title('Growth of Factorials')
plt.savefig('factorial_growth.pdf')
plt.show()

## 大きな問題
都市数を数十程度とし、位置をランダムに設定する。都市間の距離は、ユークリッド距離とする。

1. はじめにランダムな経路$[v_0, v_1, \dots, v_{n-1}]$を生成し、その経路の距離$E_\text{prev}$を計算する。
2. ２つの都市$v_i$と$v_j$をランダムに選び、その間の経路を逆転した経路$[v_0, \dots, v_{i-1}, v_j, v_{j-1}, \dots, v_i, v_{j+1}, \dots, v_{n-1}]$を生成する。この経路の距離$E_\text{new}$を計算する。
3. $E_\text{new} < E_\text{prev}$であれば、$E_\text{prev} = E_\text{new}$とし、$[v_0, \dots, v_{i-1}, v_j, v_{j-1}, \dots, v_i, v_{j+1}, \dots, v_{n-1}]$を新しい経路とする。
4. これを十分な回数繰り返す。

We will set the number of cities to a few dozen and randomly assign their positions. The distance between cities will be defined as the Euclidean distance.
1. First, we generate a random route $[v_0, v_1, \dots, v_{n-1}]$ and calculate its distance $E_\text{prev}$.
2. We randomly select two cities $v_i$ and $v_j$, and generate a new route by reversing the segment between them: $[v_0, \dots, v_{i-1}, v_j, v_{j-1}, \dots, v_i, v_{j+1}, \dots, v_{n-1}]$. We then calculate the distance of this new route, $E_\text{new}$.
3. If $E_\text{new} < E_\text{prev}$, we update $E_\text{prev} = E_\text{new}$ and set the new route as the current route.
4. We repeat this process for a sufficient number of iterations.

In [ ]:
def large_problem(num_cities:int):
    g = nx.Graph()
    cities = [f'C{i}' for i in range(num_cities)]
    edges = []
    for i in range(num_cities):
        for j in range(i + 1, num_cities):
            g.add_edge(cities[i], cities[j])
    position = {city: (random.random(), random.random()) for city in cities}
    nx.set_node_attributes(g, position, 'pos')
    for u, v in g.edges():
        g[u][v]['weight'] = math.dist(position[u], position[v])
    return g

num_cities = 40
g = large_problem(num_cities)
pos = nx.get_node_attributes(g, 'pos')
nx.draw(g, pos=pos, with_labels=True,edge_color='gray')

In [ ]:
def reverse_path(g:nx.Graph,num_iterations:int):
    tour = list(g.nodes())
    random.shuffle(tour)
    e = float('inf')
    for _ in range(num_iterations):
        i = 0
        while i ==0:
            i, j = sorted(random.sample(range(len(tour)), 2))
        new_tour = tour[:i] + tour[i:j][::-1] + tour[j:]
        cost = sum(g[new_tour[k]][new_tour[(k + 1) % len(new_tour)]]['weight'] for k in range(len(new_tour)))
        if cost < e:
            tour = new_tour
            e = cost
            print(f"New cost {e}, best tour: {tour}")
    return tour, e


best_tour, min_cost = reverse_path(g, 10000)
print("Best tour:", best_tour)
print("Minimum cost:", min_cost)
edges_in_best_tour = [(best_tour[i], best_tour[(i + 1) % len(best_tour)]) for i in range(len(best_tour))]
nx.draw(g, pos=pos, with_labels=True, edge_color='lightgray')
nx.draw_networkx_edges(g, pos=pos, edgelist=edges_in_best_tour, edge_color='r', width=2)
plt.show()

## モンテカルロ法
先程の方法では、最所の経路の選びかたに依存して、近くにある局所最小値に陥る可能性がある。モンテカルロ法では、経路を逆転する際に、距離が短くなる場合だけでなく、距離が長くなる場合も一定の確率で受け入れることにする。これにより、局所最小値から脱出できる可能性が高まる。

In the previous method, there is a possibility of getting stuck in a local minimum depending on the choice of the initial route. In the Monte Carlo method, when we reverse the route, we not only accept it if the distance is shorter but also accept it with a certain probability even if the distance is longer. This increases the chances of escaping from local minima.

In [ ]:
def monte_carlo_reverse_path(g:nx.Graph,num_iterations:int, temperature:float, cooling_rate:float, cooling_iteration:int):
    tour = list(g.nodes())
    random.shuffle(tour)
    e = float('inf')
    for _ in range(cooling_iteration):
        for k in range(num_iterations):
            i = 0
            while i ==0:
                i, j = sorted(random.sample(range(len(tour)), 2))
            new_tour = tour[:i] + tour[i:j][::-1] + tour[j:]
            cost = sum(g[new_tour[k]][new_tour[(k + 1) % len(new_tour)]]['weight'] for k in range(len(new_tour)))
            if cost < e or random.random() < math.exp((e - cost) / temperature):
                tour = new_tour
                e = cost
                print(f"New cost {e}, best tour: {tour}")
        temperature *= cooling_rate
    return tour, e

best_tour, min_cost = monte_carlo_reverse_path(g, 100, 20.0, 0.9, 5000)
print("Best tour:", best_tour)
print("Minimum cost:", min_cost)
edges_in_best_tour = [(best_tour[i], best_tour[(i + 1) % len(best_tour)]) for i in range(len(best_tour))]
nx.draw(g, pos=pos, with_labels=True, edge_color='lightgray')
nx.draw_networkx_edges(g, pos=pos, edgelist=edges_in_best_tour, edge_color='r', width=2)
plt.show()